# Notebook de predicciones con Chat GPT

## 1. Confiugración del entorno

In [28]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import pandas as pd
from tqdm.auto import tqdm
from tenacity import retry, stop_after_attempt, wait_random_exponential

# Configuración de OpenAI
client = OpenAI()
load_dotenv()  # carga variables de .env
client.api_key = os.getenv("OPENAI_API_KEY")

In [29]:
print([m.id for m in client.models.list()])   # debería mostrar modelos como 'gpt-4o'

['gpt-4-0613', 'gpt-4', 'gpt-3.5-turbo', 'gpt-4o-audio-preview-2025-06-03', 'gpt-4.1-nano-2025-04-14', 'gpt-4.1-nano', 'gpt-image-1', 'gpt-4o-realtime-preview-2025-06-03', 'davinci-002', 'babbage-002', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo-instruct-0914', 'dall-e-3', 'dall-e-2', 'gpt-4-1106-preview', 'gpt-3.5-turbo-1106', 'tts-1-hd', 'tts-1-1106', 'tts-1-hd-1106', 'text-embedding-3-small', 'text-embedding-3-large', 'gpt-4-0125-preview', 'gpt-4-turbo-preview', 'gpt-3.5-turbo-0125', 'gpt-4-turbo', 'gpt-4-turbo-2024-04-09', 'gpt-4o', 'gpt-4o-2024-05-13', 'gpt-4o-mini-2024-07-18', 'gpt-4o-mini', 'gpt-4o-2024-08-06', 'chatgpt-4o-latest', 'o1-preview-2024-09-12', 'o1-preview', 'o1-mini-2024-09-12', 'o1-mini', 'gpt-4o-realtime-preview-2024-10-01', 'gpt-4o-audio-preview-2024-10-01', 'gpt-4o-audio-preview', 'gpt-4o-realtime-preview', 'omni-moderation-latest', 'omni-moderation-2024-09-26', 'gpt-4o-realtime-preview-2024-12-17', 'gpt-4o-audio-preview-2024-12-17', 'gpt-4o-mini-realtime-preview-2

## 2. Carga de datos

In [ ]:
path_in = "../data/interim/dataset_ranking_test.csv"
df = pd.read_csv(path_in)

## 3. Preparación de Prompts

In [31]:
# Define el system prompt
system_prompt = """
Eres un evaluador automático de compatibilidad profesional. Recibirás dos textos delimitados por etiquetas claras:

CV:
<CV_CONTENT>

OFERTA:
<JOB_CONTENT>

Tarea:
1. Analiza y compara la información del CV con los requisitos de la oferta.
2. Genera un único número decimal en el rango [0, 1] que represente la adecuación del CV a la oferta:
   • 0  →  ninguna compatibilidad  
   • 1  →  encaje perfecto
3. Tu respuesta **debe consistir exclusivamente** en ese número (ejemplo: `0.87`), con punto como separador decimal.  
   • No añadas texto, explicaciones, etiquetas, unidades, ni formato JSON.  
   • No incluyas saltos de línea adicionales antes ni después del número.

Criterios (razona internamente, no los muestres):
- Coincidencia de habilidades técnicas y blandas.
- Nivel y años de experiencia requeridos.
- Educación, certificaciones e idiomas.
- Sector, ubicación y otros requisitos específicos.

Piensa paso a paso de forma privada y responde solo con el número final.
""".strip()

## 4. Lllamadas a la API

In [30]:
# Modelo provisional para pruebas
MODELO_DEV = "gpt-3.5-turbo"

# Modelo final para inferencia validada
MODELO_PROD = "gpt-4o"


In [32]:
@retry(stop=stop_after_attempt(6), wait=wait_random_exponential(multiplier=1, max=20))
def get_match_score(cv_text: str, oferta_text: str, model_name: str = MODELO_DEV) -> float:
    """Devuelve la compatibilidad [0,1] entre un CV y una oferta."""
    user_prompt = f"CV:\n{cv_text}\n\nOFERTA:\n{oferta_text}"
    resp = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0,
    )
    return float(resp.choices[0].message.content.strip())

## 5. Evaluación de los pares CV-Oferta

**Inferencia sobre los pares CV-Oferta**

A continuación, iteramos sobre todos los pares (CV, oferta) del conjunto de test y evaluamos la adecuación usando el modelo `gpt-4o` vía la API de OpenAI.

- Cada llamada devuelve un `score` entre 0 y 1.
- El resultado se almacena en la nueva columna `score_pred`.
- Este paso puede tardar varios minutos dependiendo del número de pares y la latencia de la API.


In [34]:
scores = []
for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluando pares"):
    scores.append(get_match_score(row["texto_cv"], row["texto_oferta"], model_name=MODELO_PROD))

df["score_pred"] = scores


Evaluando pares:   0%|          | 0/285 [00:00<?, ?it/s]

## 6. Ranking y guardado

**Generación del ranking final**

Se calcula el ranking de cada oferta para un CV dado, ordenando las puntuaciones `score_pred` de mayor a menor.

- El ranking se guarda como un entero en la columna `rank`.
- Esto nos permite obtener directamente el top-4 por cada CV para las métricas de evaluación posteriores.


In [35]:
df["rank"] = (
    df.groupby("cv_id")["score_pred"]
      .rank(method="first", ascending=False)
      .astype(int)
)


In [36]:
out_cols = ["cv_id", "offer_id", "score_pred", "rank"]

(
    df[out_cols]                # selecciona columnas
      .sort_values(["cv_id",    # primero por cv…
                    "rank"])    # …luego por rank ascendente
      .to_csv("../reports/rankings_test_chatgpt.csv", index=False)
)
print("✅ Archivo ordenado por cv_id y rank guardado correctamente.")

✅ Archivo ordenado por cv_id y rank guardado correctamente.


**Guardado de resultados**

El DataFrame final se guarda en el directorio `../reports` con el nombre `rankings_test_chatgpt.csv`.

Este archivo se utilizará para la evaluación final del modelo con métricas como NDCG@4, MAP@4 y Overlap@4.
